# 01 — Catálogo de subtipos e definição do recorte de poda

**Pergunta:** como a "poda de árvore" aparece na taxonomia de chamados do 1746 e quais
subtipos devem compor o recorte da análise?

> As saídas abaixo são as da execução original no Google Colab (dados extraídos do BigQuery em novembro de 2026). Para reproduzir, veja o README.

In [ ]:
# Configuração (rode primeiro). Defina GCP_PROJECT_ID antes; veja o README.
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from google.cloud import bigquery

from podas.bq import get_client, rodar
from podas.config import ALVO, SENSIBILIDADE, ARQ_PODA, DATA_DIR, FIG_DIR, OUT_DIR

client = get_client()

## 1. Esquema da tabela `chamado`

In [ ]:
sql = """
SELECT column_name, data_type
FROM `datario.adm_central_atendimento_1746.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'chamado'
ORDER BY ordinal_position
"""
df = client.query(sql).to_dataframe()
df

                         column_name  data_type
0                         id_chamado     STRING
1               id_origem_ocorrencia     STRING
2                        data_inicio   DATETIME
3                           data_fim   DATETIME
4                          id_bairro     STRING
5                id_territorialidade     STRING
6                      id_logradouro     STRING
7                  numero_logradouro      INT64
8          id_unidade_organizacional     STRING
9        nome_unidade_organizacional     STRING
10     id_unidade_organizacional_mae     STRING
11  unidade_organizacional_ouvidoria     STRING
12                         categoria     STRING
13                           id_tipo     STRING
14                              tipo     STRING
15                        id_subtipo     STRING
16                           subtipo     STRING
17                            status     STRING
18                         longitude    FLOAT64
19                          latitude    

## 2. Subtipos que mencionam poda ou árvore (contagem por ano)

In [ ]:
sql = """
SELECT
  tipo,
  subtipo,
  EXTRACT(YEAR FROM data_inicio) AS ano,
  COUNT(*) AS n_chamados
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE LOWER(CONCAT(IFNULL(tipo,''), ' ', IFNULL(subtipo,''))) LIKE '%poda%'
   OR LOWER(CONCAT(IFNULL(tipo,''), ' ', IFNULL(subtipo,''))) LIKE '%árvore%'
   OR LOWER(CONCAT(IFNULL(tipo,''), ' ', IFNULL(subtipo,''))) LIKE '%arvore%'
GROUP BY tipo, subtipo, ano
ORDER BY tipo, subtipo, ano
"""

# estimar o custo (não consome cota)
cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
print(f"{client.query(sql, job_config=cfg).total_bytes_processed/1e9:.2f} GB")

# rodar com teto de segurança
cfg = bigquery.QueryJobConfig(maximum_bytes_billed=20 * 10**9)
tipos = client.query(sql, job_config=cfg).to_dataframe()
tipos

1.54 GB


                  tipo                                            subtipo  \
0            Acidentes  Envio da Guarda Municipal nas situações de Tom...   
1            Acidentes  Envio da Guarda Municipal nas situações de Tom...   
2          Arborização  Autorização de poda ou remoção de árvore em ár...   
3          Arborização  Autorização de poda ou remoção de árvore em ár...   
4          Arborização  Autorização de poda ou remoção de árvore em ár...   
..                 ...                                                ...   
305  Árvores - Plantio                   Árvores - assuntos gerais da FPJ   
306  Árvores - Plantio                   Árvores - assuntos gerais da FPJ   
307  Árvores - Plantio                   Árvores - assuntos gerais da FPJ   
308  Árvores - Plantio                   Árvores - assuntos gerais da FPJ   
309  Árvores - Plantio                   Árvores - assuntos gerais da FPJ   

      ano  n_chamados  
0    2011          47  
1    2012          69  
2  

In [ ]:
print(
    tipos.groupby(["tipo", "subtipo"])["n_chamados"]
    .sum()
    .sort_values(ascending=False)
    .to_string()
)

tipo                                                          subtipo                                                                             
Manejo Arbóreo                                                Poda de árvore em logradouro                                                            343765
                                                              Remoção de árvore em logradouro                                                          82664
                                                              Avaliação de risco de queda da árvore                                                    68122
Limpeza de logradouros                                        Remoção de resíduos de poda                                                              65303
Manejo Arbóreo                                                Árvore-Poda e Remoção                                                                    46326
                                                              Poda d

O tipo **Manejo Arbóreo** concentra quase todo o volume. Há subtipos quase idênticos
("Poda de árvore em logradouro", "Poda de árvore em vias públicas", "Árvore - Poda",
"Árvore-Poda e Remoção"), sinal de mudanças na taxonomia do 1746 ao longo do tempo.

## 3. Catálogo completo e busca por palavras-chave

In [ ]:
sql = """
SELECT tipo, subtipo, COUNT(*) AS n
FROM `datario.adm_central_atendimento_1746.chamado`
GROUP BY tipo, subtipo
"""

# estimar custo
cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
print(f"{client.query(sql, job_config=cfg).total_bytes_processed/1e9:.2f} GB")

# rodar
cfg = bigquery.QueryJobConfig(maximum_bytes_billed=20 * 10**9)
todos = client.query(sql, job_config=cfg).to_dataframe()

print(f"soma dos grupos: {todos['n'].sum():,}")
print(f"esperado:        {20263796:,}")
print(f"subtipos distintos: {len(todos):,}")

1.38 GB
soma dos grupos: 20,263,796
esperado:        20,263,796
subtipos distintos: 5,545


A soma dos grupos confere com o total da tabela (20.263.796 chamados, 5.545 combinações tipo/subtipo).

In [ ]:
import re

# 1) subtipos com menção a galho, fiação, rede elétrica, poste, Light etc.
padrao = re.compile(r"galho|vegeta|ro[cç]ada|copa|fia[cç][aã]o|fio|rede el[eé]trica|poste|light", re.I)
extra = todos[todos["subtipo"].fillna("").str.contains(padrao)
              | todos["tipo"].fillna("").str.contains(padrao)]
print("=== BUSCA POR PALAVRAS-CHAVE ===")
print(extra.sort_values("n", ascending=False).head(40).to_string())

# 2) os cinco subtipos de poda por ano (quebra de taxonomia)
ALVO = [
    "Poda de árvore em logradouro",
    "Poda de árvore em vias públicas",
    "Árvore - Poda",
    "Árvore-Poda e Remoção",
    "Poda e remoção de árvore",
]
p = (tipos[tipos["subtipo"].isin(ALVO)]
     .pivot_table(index="ano", columns="subtipo", values="n_chamados",
                  aggfunc="sum", fill_value=0))
print("\n=== PODA POR ANO ===")
print(p.to_string())

# 3) colunas reais da tabela (não gasta cota)
tab = client.get_table("datario.adm_central_atendimento_1746.chamado")
print("\n=== COLUNAS ===")
for c in tab.schema:
    print(c.name, c.field_type)

=== BUSCA POR PALAVRAS-CHAVE ===
                                  tipo                                                                                                                                                                                        subtipo      n
4366                Iluminação Pública                                                                                                                                                                   Reparo de poste alto apagado  60422
3205                Iluminação Pública                                                                                                                                                     Reparo de poste caído ou em risco de queda  23874
3210            Danos ao meio ambiente                                                                                                                            Fiscalização de corte, sacrifício de árvore ou remoção de vegetação  22326
4404               

## 4. Classificação dos subtipos: alvo, sensibilidade, revisar e fora

In [ ]:
import re, unicodedata

def norm(s):
    return unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().lower()

ALVO = [
    "Poda de árvore em logradouro",
    "Poda de árvore em vias públicas",
    "Árvore - Poda",
]
SENSIBILIDADE = ["Poda e remoção de árvore"]

amplo = re.compile(
    r"arvor|poda|galho|vegeta|tronco|folhag|rocad|capina|fiacao|\bfios?\b|cabo|"
    r"rede (eletrica|aerea)|\blight\b|energia|eletric|tombament|queda"
)

cat = todos.copy()
cat["txt"] = cat["tipo"].map(norm) + " | " + cat["subtipo"].map(norm)

def classe(r):
    if r["subtipo"] in ALVO: return "alvo"
    if r["subtipo"] in SENSIBILIDADE: return "sensibilidade"
    if amplo.search(r["txt"]): return "revisar"
    return "fora"

cat["classe"] = cat.apply(classe, axis=1)

resumo = cat.groupby("classe")["n"].agg(["count", "sum"])
print(resumo)
assert resumo["sum"].sum() == 20_263_796, "a partição perdeu linhas"

# tudo que entrou em 'revisar' e 'fora' em arquivos para você ler
cat[cat.classe == "revisar"].sort_values("n", ascending=False).to_csv(OUT_DIR / "revisar.csv", index=False)
cat[cat.classe == "fora"].sort_values("n", ascending=False).to_csv(OUT_DIR / "fora.csv", index=False)

print(cat[cat.classe == "revisar"].sort_values("n", ascending=False).head(60)[["tipo","subtipo","n"]].to_string())
print("\nMaiores itens de 'fora':")
print(cat[cat.classe == "fora"].sort_values("n", ascending=False).head(80)[["tipo","subtipo","n"]].to_string())

               count       sum
classe                        
alvo               3    376293
fora            5449  19285698
revisar           92    588115
sensibilidade      1     13690
                                                              tipo                                                                                             subtipo       n
1449                                        Limpeza de logradouros                                                                                Capina em logradouro  158610
4031                                                Manejo Arbóreo                                                                     Remoção de árvore em logradouro   82664
902                                                 Manejo Arbóreo                                                               Avaliação de risco de queda da árvore   68122
2032                                        Limpeza de logradouros                                                

In [ ]:
import pandas as pd

colunas = ["classe", "tipo", "subtipo", "n"]

abas = {
    "alvo":    cat[cat.classe.isin(["alvo", "sensibilidade"])],
    "revisar": cat[cat.classe == "revisar"],
    "fora":    cat[cat.classe == "fora"],
}

with pd.ExcelWriter(OUT_DIR / "classificacao_subtipos.xlsx", engine="openpyxl") as xl:
    for nome, df in abas.items():
        df = df.sort_values("n", ascending=False)[colunas]
        df.to_excel(xl, sheet_name=nome, index=False)

        ws = xl.sheets[nome]
        ws.freeze_panes = "A2"                # cabeçalho fixo
        ws.auto_filter.ref = ws.dimensions    # filtros nas colunas
        for col, larg in zip("ABCD", [14, 45, 90, 12]):
            ws.column_dimensions[col].width = larg
        for cell in ws["D"][1:]:
            cell.number_format = "#,##0"      # milhar no n

print({k: len(v) for k, v in abas.items()})

{'alvo': 4, 'revisar': 92, 'fora': 5449}


**Decisão de recorte**

- **Alvo:** `Poda de árvore em logradouro`, `Poda de árvore em vias públicas` e `Árvore - Poda`.
- **Sensibilidade:** `Poda e remoção de árvore` (mistura poda e remoção; fica fora do recorte principal).
- `Árvore-Poda e Remoção` fica de fora porque não tem registros em 2021–2025.

A partição das 5.545 combinações foi conferida (`assert`): nenhuma linha se perdeu.

## 5. Subtipos de Manejo Arbóreo no período de estudo (2021–2025)

In [ ]:
sql_tipo = """
SELECT subtipo,
       COUNT(*) AS total,
       COUNTIF(data_inicio >= DATETIME '2021-01-01'
               AND data_inicio <  DATETIME '2026-01-01') AS n_2021_2025
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE tipo = 'Manejo Arbóreo'
GROUP BY subtipo
ORDER BY total DESC
"""
tipo_df = rodar(sql_tipo)
print(tipo_df.to_string(index=False))
print("TOTAL manejo arbóreo:", tipo_df["total"].sum(),
      "| 2021-2025:", tipo_df["n_2021_2025"].sum())

custo estimado: 1.525 GB
                                   subtipo  total  n_2021_2025
              Poda de árvore em logradouro 276022       123744
           Poda de árvore em vias públicas  98817         9143
     Avaliação de risco de queda da árvore  68122        20357
           Remoção de árvore em logradouro  65076        29204
                     Árvore-Poda e Remoção  46326            0
        Remoção de árvore em vias públicas  20206          142
          Remoção de toco e raiz (destoca)  20120         7546
     Reparo de calçada danificada por raiz  15809         5591
                  Poda e remoção de árvore  13690         5655
                  Retirada de árvore caída  11144         7890
                     Reparação pós-destoca   6205         1977
                             Árvore - Poda   1454          828
                  Árvore - assuntos gerais    322          230
                          Árvore - Remoção    285          162
Auto de infração - constatação

`Poda de árvore em vias públicas` tem 9.143 chamados em 2021–2025, mas 21.780 já em 2026:
o 1746 parece estar migrando a poda para esse subtipo. Isso precisa ser considerado ao
estender a análise para 2026.

## 6. Subtipos de queda / risco (candidatos a desfecho)

In [ ]:
sql_cat = """
SELECT tipo, subtipo,
       COUNT(*) AS n_total,
       COUNTIF(data_inicio >= DATETIME '2021-01-01' AND data_inicio < DATETIME '2026-01-01') AS n_2021_2025,
       COUNTIF(data_inicio >= DATETIME '2026-01-01') AS n_2026,
       MIN(EXTRACT(YEAR FROM data_inicio)) AS ano_ini,
       MAX(EXTRACT(YEAR FROM data_inicio)) AS ano_fim
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE REGEXP_CONTAINS(LOWER(CONCAT(IFNULL(tipo, ''), ' ', IFNULL(subtipo, ''))),
                      r'qued|ca[ií]d|desab|tomb|galho|árvore|arvore')
GROUP BY tipo, subtipo
ORDER BY n_2021_2025 DESC
"""
cat = rodar(sql_cat)
print(cat.to_string(index=False))

custo estimado: 1.525 GB
                                                        tipo                                                                                                                 subtipo  n_total  n_2021_2025  n_2026  ano_ini  ano_fim
                                              Manejo Arbóreo                                                                                            Poda de árvore em logradouro   276022       123744       0     2014     2025
                                              Manejo Arbóreo                                                                                         Remoção de árvore em logradouro    65076        29204    1588     2014     2026
                                              Manejo Arbóreo                                                                                   Avaliação de risco de queda da árvore    68122        20357    4446     2011     2026
                                         Estrutura de Imóve

Para a análise poda → queda (notebook 05), o desfecho escolhido é
**`Retirada de árvore caída`** (Manejo Arbóreo): 7.890 chamados em 2021–2025, registrado desde 2019.